# Colab: forecast model (water levels 2025-2027)

5 runs, about 2.5 hours. The final model (model-v1.0 settings, nothing changed) is retrained on **every** well reading up to 2024, and its physics runs to the end of 2027 with the latest IMD rainfall. This gives the website forecast maps for 2025-2027.

**Do not change any cell.** Just follow these steps:
1. Put the **new** `pinn_code.zip` (from the project lead) in a Google Drive folder called `pinn`, so you have `MyDrive/pinn/pinn_code.zip`. Replace any older copy.
2. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Allow Google Drive access when asked.
3. Keep the laptop awake (plugged in, sleep off, this tab open). If Colab disconnects, just **Run all** again: finished runs are skipped.
4. When the runs are done, run the **last cell** and send `MyDrive/pinn/forecast_results.zip` to the project lead.


In [ ]:
SEEDS = [0, 1, 2, 3, 4]
FORECAST_END = "2027-12-31"
DRIVE_ROOT = "/content/drive/MyDrive/pinn"


In [ ]:
import glob
import os
import shutil
import zipfile
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(f"{DRIVE_ROOT}/runs/logs", exist_ok=True)
CODE = "/content/pinn"
shutil.rmtree(CODE, ignore_errors=True)
with zipfile.ZipFile(f"{DRIVE_ROOT}/pinn_code.zip") as z:
    z.extractall(CODE)
%cd {CODE}
assert os.path.exists("data/processed/rain_daily_forecast.nc"), "old pinn_code.zip: ask the project lead for the new one"


In [ ]:
# PyTorch is already installed on Colab; this adds everything else (about 1 minute).
!pip install -q -r requirements.txt
!pip install -q -e .
os.environ["PINN_RUNS_DIR"] = f"{DRIVE_ROOT}/runs"
!python scripts/check_env.py
!python -m pytest -q -m "not data"


## Runs
The shared zoned PINN with canals (model-v1.0 settings), 5 seeds, trained on all wells and readings, physics to the end of 2027.


In [ ]:
def done(pattern):
    return bool(glob.glob(f"{DRIVE_ROOT}/runs/{pattern}/metrics.json"))

LOG = f"{DRIVE_ROOT}/runs/logs/forecast.log"
for seed in SEEDS:
    if done(f"*_real_L0_kmeansk5_shared_s{seed}_fc"):
        print("already done: seed", seed)
        continue
    !python -u scripts/run_real.py --shared --layout 0 --k 5 --seed {seed} --tag fc --forecast-end {FORECAST_END} 2>&1 | tee -a {LOG}


## Bundle the results for Claude
Makes `MyDrive/pinn/forecast_results.zip` with every **finished** run (run it any time).
Download it, put it in the `pinn` folder on the laptop, then tell Claude.


In [ ]:
out = f"{DRIVE_ROOT}/forecast_results.zip"
n = 0
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for run in glob.glob(f"{DRIVE_ROOT}/runs/*_real_L0_kmeansk5_shared_s*_fc"):
        if not os.path.exists(f"{run}/metrics.json"):
            continue
        n += 1
        for path in glob.glob(f"{run}/*"):
            z.write(path, os.path.relpath(path, f"{DRIVE_ROOT}/runs"))
    if os.path.exists(LOG):
        z.write(LOG, "logs/forecast.log")
print("wrote", out, f"({n} finished runs, {os.path.getsize(out) / 1e6:.2f} MB)")
